**1D**
1) Define the domain $\Omega$
2) Generate the mesh
3) Get the step h
4) Get stiffness matrix A
5) Get mass matrix M
6) Implement Dirichlet boundary condition
7) Solve $A * c = \lambda * M * c$
8) get the eigenvalues
9) get the eigenvectors
10) Compare them to the exact solution

**2D**
1) Define the domain $\Omega$
2) Generate triangulations T_h
3) Construct finite element space V_h
4) Get local stiffness matrix A_k
5) Get local mass matrix M_k
6) Assemble local stiffness matrices into global stiffness matrix A
7) Assemble local mass matrices into a global mass matrix M
8) Impose Dirichlet Boundary Condition
9) Solve $A * u = \lambda * M * u$
10) Extract eigenvalues
11) compute the eigenvectors
12) Compare with exact solution


In [22]:
# 1D let the domain be uniform and defined on [0, 1]
import numpy as np
from scipy.linalg import eigh
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [28]:

# method for putting values on the main diagonal
def off_diag_values_matrix(rank, value):
       # the size should be smaller because there are less numbers off diagonal than on main diagonal
       diag = np.full(rank - 1, value)
       lower_off_centre_diag = np.diag(diag, k=-1)
       higher_off_centre_diag = np.diag(diag, k=1)

       matrix = lower_off_centre_diag + higher_off_centre_diag
       

       return matrix

# stiffness matrix
def stiffness_matrix_A(rank, h):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       A = np.zeros((rank, rank), dtype=float)
       # diagonal values
       a_ii = 2 / h
       np.fill_diagonal(A, a_ii)
       # off diagonal values
       a_ij = -1 / h

       A = A + off_diag_values_matrix(rank, a_ij)
       return A

# mass matrix
def mass_matrix_M(rank, h):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       M = np.zeros((rank, rank), dtype=float)
       # diagonal values
       m_ii = (4 * h) / 6
       np.fill_diagonal(M, m_ii)
       # off diagonal values
       m_ij = h / 6

       M = M + off_diag_values_matrix(rank, m_ij)
       return M

# error of all eigenvalues
# takes 2 lists of real eigenvalues and computed
def all_eigval_error(real_eigvals, comp_eigvals):
       # for values in both arrays
       for real_val, val in zip(real_eigvals, comp_eigvals):
              error = abs(real_val - val)
              print(
                     f"Computed eigenvalue: {val:.5f}\n",
                     f"Real value: {real_val:.5f}\n",
                     f"Error: {error:.5f}"
              )

# error of the first eigenvalue
# will get only the first elements of the arrays
def first_eigval_error(real_eigval, comp_eigval):
       error = abs(real_eigval - comp_eigval)
       return error

def make_table(headers, indexes, table):
       
       df = pd.DataFrame(table, columns = headers, index = indexes)
       return df

def plotting_convergence(h_vals, errors):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(h_vals, errors, marker="o")
       ax.set_xlabel("x")
       ax.set_ylabel(r"$E(h) = |\lambda_1 - \lambda_1^h|$")
       ax.set_title("Convergence of the FEM eigenvalues")
       ax.grid(True)

def plotting_eigenfunction(x, u):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(x, u, marker="o")
       ax.set_xlabel("x")
       ax.set_ylabel(r"$\phi_1^h(x)$")
       ax.set_title("First Eigenfunction")
       ax.grid(True)

def comparing_exact_with_comp_functions(x, u_comp, x_of_exact):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(x, u_comp, "b", label="Computed function")
       ax.plot(x_of_exact, np.sin(np.pi*x_of_exact), "ro", label="Exact function")
       ax.set_xlabel("x")
       ax.set_ylabel("u(x)")
       ax.legend()
       ax.grid(True)
       



In [ ]:

x_0 = 0
x_n = 1
N = int(input("How many elements do you want to have: "))
try: 
       h = 1 / N
except ZeroDivisionError:
       print("You cannot have 0 elements")

nodes = N + 1
interior_nodes = N - 1 # because of Dirichlet boundary condition 0 element and n-th element will be zeros


# get eigenvalues and eigenvectors
A = stiffness_matrix_A(interior_nodes, h)
M = mass_matrix_M(interior_nodes, h)
# get the eigenvalues and eigenvectors
# it will give 2 arrays, one for eigenvalues, another for eigenvectors
eigvals, eigvecs = eigh(A, M)

# real eigenvalues
real_eigvals = np.array([(np.pi * i)**2 for i in range(1, len(eigvals) + 1)])

# error check for all eigenvalues
all_eigval_error(real_eigvals, eigvals)

# error of the first eigenvalue
first_error = first_eigval_error(real_eigvals[0], eigvals[0])
print(first_error)

In [ ]:
print("\nSanity check: check first eigenvalues for N = 2, 4, 8, 16\n")
N = [2, 4, 8, 16]
errors = []
rows = []
h_vals = []

for num_elements in N:
       row = []
       row.append(num_elements)
       # find the step
       h = 1 / num_elements
       row.append(h)
       h_vals.append(h)
       # get the interior nodes
       interior_nodes = num_elements - 1

       # stiffness matrix
       A = stiffness_matrix_A(interior_nodes, h)
       M = mass_matrix_M(interior_nodes, h)
       # get the eigenvalues and eigenvectors
       # it will give 2 arrays, one for eigenvalues, another for eigenvectors
       eigvals, eigvecs = eigh(A, M)
       row.append(float(eigvals[0]))

       # real eigenvalue
       real_eigval = (np.pi)**2
       row.append(float(real_eigval))

       # first eigenvalue error
       error = first_eigval_error(real_eigval, eigvals[0])
       errors.append(error)
       row.append(float(error))
       rows.append(row)
       print(f"The error of the first eigenvalue = {error}")

# convergence rate
# local convergence rate

# create an array of log2(e_i/e_i+1) 
# so this array checks the local convergence rate between 2-4, 4-8 and 8-16 
p_local = [np.log2(errors[i-1] / errors[i]) for i in range(1, len(errors))]
p_local_np = np.array(p_local)
# global convergence rate
p = np.mean(p_local)

# make table
headers = ["N", "h", r"$\lambda_1$", r"$\lambda_real$", "error"]
indexes = ["1", "2", "3", "4"]

table = make_table(headers, indexes, rows)
print(table)
plotting_convergence(h_vals, errors)
